# 07 — Local Green, Global Red in LLM and agentic pipelines

> **Claim-typing legend**
> - **[Pinned]**: produced by `code/epk_core.py` and named tests.
> - **[Proved]**: a named Lean theorem in `lean4/` (`PaperAE004`).
> - **[Declared]**: a teaching model or convention. Not forced.
> - **[Open]**: recorded open in the ledger (empirical frontier-LLM validation).

This notebook is the LLM / agentic reading of Paper 012. It uses the **same**
shipped engine as the Five-Trace Phone Test. It does **not** measure GPT-class
models.

**A receipt is never its payload.** A green cell means the arithmetic ran.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

nb_dir = Path.cwd().resolve()
if (nb_dir / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir))
elif (nb_dir / "jupyter_notebooks" / "epk_nb.py").is_file():
    sys.path.insert(0, str(nb_dir / "jupyter_notebooks"))
from epk_nb import import_epk, widgets_available

epk = import_epk()
HAS_WIDGETS = widgets_available()

from epk_core import *  # shipped names; notebooks must not reimplement the engine

print("Python ready.")
print("epk_core imported from shipped code/ (filesystem path omitted).")
print("HAS_WIDGETS =", HAS_WIDGETS)


In [ ]:
def explain_delta(e1, e2, eg):
    d = calculate_evaluator_defect(int(e1), int(e2), int(eg))
    kind = {
        -1: "NGD — local steps green, composed task red",
        0: "ZDR — local product agrees with global",
        1: "PGD — local product red, global green",
    }[d]
    print(f"E1={int(e1)}  E2={int(e2)}  Eg={int(eg)}  =>  Delta_E={d}  ({kind})")
    return d

if HAS_WIDGETS:
    import ipywidgets as widgets
    from IPython.display import display
    print("Interactive: flip local and global verdicts. Headless runs skip this widget.")
    widgets.interact(explain_delta, e1=[0, 1], e2=[0, 1], eg=[0, 1])
else:
    print("Headless fallback — every Boolean triple:")
    for e1 in (0, 1):
        for e2 in (0, 1):
            for eg in (0, 1):
                explain_delta(e1, e2, eg)


## 1. The mapping, in one table [Declared]

Think of an agentic run as a composed trajectory \(\gamma = \gamma_2 \star \gamma_1\).

| Paper object | LLM / agentic reading |
|---|---|
| Local evaluator \(E_1\) | Did step 1 pass its *own* check? (schema valid, tool returned 200, unit test green) |
| Local evaluator \(E_2\) | Did step 2 pass *its* check? (patch compiles, linter clean, SQL parses) |
| Global evaluator \(E_g\) | Did the *task* succeed? (integration test, user-visible goal, SWE-bench instance) |
| \(\Delta_E = E_g - E_1 E_2\) | Did local greens compose? |
| NGD (\(\Delta_E=-1\)) | Both steps look fine; the job is still wrong. **Local green, global red.** |
| ZDR (\(\Delta_E=0\)) | Local product and global verdict agree (both fail, or both succeed). |
| PGD (\(\Delta_E=+1\)) | A local check failed, yet the task still passed (rescue / skipped gate). |
| Detector adequacy | Are the local checks even capable of failing? Vacuous unit tests force fake NGD. |

[Pinned] The Phone Test is this table with five hand traces. Same numbers, same functions.


## 2. A tiny agent story [Declared]

Two agents work on a repo:

1. **Planner** writes an interface change. Local check: "the spec JSON is valid." \(E_1=1\).
2. **Coder** implements against yesterday's interface. Local check: "the file compiles." \(E_2=1\).
3. **CI** runs the product test. It fails. \(E_g=0\).

That is NGD: \(\Delta_E = 0 - 1\cdot 1 = -1\). [Pinned] `calculate_evaluator_defect(1, 1, 0) == -1`. [Proved] Lean `local_green_global_red_counterexample`.

The same arithmetic covers:

- a tool-using agent whose `read_file` and `search_replace` both "succeeded" while the user goal did not;
- a multi-hop RAG chain whose retrieval and citation format both passed while the answer was false;
- a SWE-bench style patch that is locally well-formed and globally failing.

None of those stories is an empirical score. They are **the same finite identity**.


In [ ]:
phone = make_five_trace_phone_test()
report = EPKMetrics.compute(phone)
assert report.lpr == 0.6
assert report.gpr == 0.2
assert report.mean_delta_e == -0.4
assert report.ngd_all + report.pgd + report.zdr == 1.0
assert abs((report.gpr - report.lpr) - report.mean_delta_e) < 1e-12
print("Phone Test via shipped epk_core:")
print("  LPR", report.lpr, "GPR", report.gpr, "mean Delta_E", report.mean_delta_e)
print("  NGD", report.ngd_all, "PGD", report.pgd, "ZDR", report.zdr)


## 3. Picture: local greens vs the task [Pinned]

The bar chart uses **only** `make_five_trace_phone_test()`. Read it as five
agent runs. Traces 2 and 3 are the scary ones: both local gates green, task red.


In [ ]:
phone = make_five_trace_phone_test()
labels = [f"run {i+1}" for i in range(5)]
e1 = [t.e1 for t in phone]
e2 = [t.e2 for t in phone]
eg = [t.eg for t in phone]
delta = [t.delta_eval for t in phone]

fig, ax = plt.subplots(figsize=(8, 3.8))
x = np.arange(5)
w = 0.2
ax.bar(x - w, e1, w, label="E1 local step 1", color="#5cb85c")
ax.bar(x, e2, w, label="E2 local step 2", color="#5bc0de")
ax.bar(x + w, eg, w, label="Eg task / CI", color="#d9534f")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(-0.05, 1.35)
ax.set_ylabel("verdict (0 or 1)")
ax.set_title("Five agent runs: local checks vs the task")
ax.legend(loc="upper right", fontsize=8)
for i, d in enumerate(delta):
    if d == -1:
        ax.annotate("NGD", (i, 1.12), ha="center", color="#841a18", fontsize=9)
plt.show()
print("Delta_E per run:", delta)


## 4. Why a dashboard of "tool success rate" lies [Proved]

If you only plot local pass rate, you cannot see NGD. Lean proves that the
same local-pass count can hide different global counts (`E3` in
`07_dashboard_insufficiency.lean`). Python agrees:

Two samples, same LPR, different GPR.


In [ ]:
clean = [EvaluatorVerdictTriple(1, 1, 1)]
lggr = [EvaluatorVerdictTriple(1, 1, 0)]
r_clean = EPKMetrics.compute(clean)
r_lggr = EPKMetrics.compute(lggr)
assert r_clean.local_pass_count == r_lggr.local_pass_count == 1
assert r_clean.gpr != r_lggr.gpr
print("same local pass count =", r_clean.local_pass_count)
print("clean GPR", r_clean.gpr, "LGGR GPR", r_lggr.gpr, "NGD", r_lggr.ngd_all)


## 5. Detector adequacy for agents [Pinned]

If every local check is hard-wired to pass, NGD becomes a costume for ordinary
task failure. `DetectorAdequacyGuard.audit` refuses that collapse.

[Declared] This is why "the linter is always green" is not evidence that the
agent is composing.


In [ ]:
vacuous = [EvaluatorVerdictTriple(1, 1, 0)] * 4
flags = DetectorAdequacyGuard.audit(vacuous)
assert DetectorAdequacyStatus.VACUOUS_LOCAL in flags
print("vacuous local gates:", [f.name for f in flags])

honest = make_five_trace_phone_test()
print("phone-test flags:", [f.name for f in DetectorAdequacyGuard.audit(honest)])


## 6. CAN-show / CANNOT-show

**CAN show**

- The identity \(\mathrm{GPR}-\mathrm{LPR}=\mathrm{mean}\,\Delta_E\) on any finite sample of binary verdicts.
- That local-step green plus local-step green need not imply task green.
- That vacuous or tautological evaluators make the dashboard uninformative.
- How to *talk* about planner/coder/CI, tool-call chains, and RAG hops in this language.

**CANNOT show**

- That any named frontier LLM fails at a measured NGD rate. `[Open]` / `[CONJECTURE]` (Prediction 1).
- That \(\Delta_E=0\) means the answer is true, aligned, or authorized.
- Quantum, Born-rule, or thermodynamic readings of token probabilities.
- Authority. Evaluators are declared finite verdicts, not a descent packet.


## 7. Student exercises

### Exercise 1
An agent: retrieval \(E_1=1\), citation formatter \(E_2=1\), fact-check \(E_g=0\).
What is \(\Delta_E\), and which defect kind?

<details><summary>Answer</summary>
\(\Delta_E = 0 - 1\cdot 1 = -1\), NGD. Same as Phone Test traces 2 and 3.
</details>

### Exercise 2
Every unit test in a repo is `assert True`. Integration still fails half the time.
Which adequacy flag fires, and why is NGD not a "composition bug" here?

<details><summary>Answer</summary>
`VACUOUS_LOCAL`. Local greens are not discriminating, so NGD only restates
global failure. Detector adequacy is the fence.
</details>


## Anchors

- Paper: `lggr_i_evaluated_path_kernels.md` §§1–2, §5, §8–§9
- Ledger: B1, B2, B3 (scope fence), E2, E3
- Python: `make_five_trace_phone_test`, `EPKMetrics.compute`, `calculate_evaluator_defect`, `DetectorAdequacyGuard.audit`
- Tests: `test_canonical_phone_test_exact_values`, `test_detector_adequacy_audits`
- Lean: `local_green_global_red_counterexample`, `phone_identity_global_minus_local`, `same_local_pass_count_different_global_ngd_and_delta`
- Map: `docs/lean_claim_map.md`
